<a href="https://colab.research.google.com/github/maibubhukya/BDA_Assignment_02-314/blob/main/BDA_Assignment_02_314.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Question 1
#1.Build a Classification with Spark with a dataset of your choice

In [3]:
# Clean old installs
!rm -rf spark-3.4.1-bin-hadoop3

# Install Java 11
!apt-get install openjdk-11-jdk-headless -qq > /dev/null

# Download Spark
!wget -q https://archive.apache.org/dist/spark/spark-3.4.1/spark-3.4.1-bin-hadoop3.tgz
!tar xf spark-3.4.1-bin-hadoop3.tgz

# Install findspark
!pip install -q findspark

# Set environment variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.4.1-bin-hadoop3"

# Start Spark
import findspark
findspark.init()

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("BDA_Lab").master("local[*]").getOrCreate()

spark

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import pandas as pd
url ="/content/heart.csv"
pdf = pd.read_csv(url)
pdf.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [7]:
df = spark.createDataFrame(pdf)
df.printSchema()
df.show(5)

root
 |-- age: long (nullable = true)
 |-- sex: long (nullable = true)
 |-- cp: long (nullable = true)
 |-- trestbps: long (nullable = true)
 |-- chol: long (nullable = true)
 |-- fbs: long (nullable = true)
 |-- restecg: long (nullable = true)
 |-- thalach: long (nullable = true)
 |-- exang: long (nullable = true)
 |-- oldpeak: double (nullable = true)
 |-- slope: long (nullable = true)
 |-- ca: long (nullable = true)
 |-- thal: long (nullable = true)
 |-- target: long (nullable = true)

+---+---+---+--------+----+---+-------+-------+-----+-------+-----+---+----+------+
|age|sex| cp|trestbps|chol|fbs|restecg|thalach|exang|oldpeak|slope| ca|thal|target|
+---+---+---+--------+----+---+-------+-------+-----+-------+-----+---+----+------+
| 63|  1|  3|     145| 233|  1|      0|    150|    0|    2.3|    0|  0|   1|     1|
| 37|  1|  2|     130| 250|  0|      1|    187|    0|    3.5|    0|  0|   2|     1|
| 41|  0|  1|     130| 204|  0|      0|    172|    0|    1.4|    2|  0|   2|     1|
| 

In [8]:
from pyspark.ml.feature import VectorAssembler

feature_cols = df.columns[:-1]   # last column is target
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

data = assembler.transform(df)
data = data.select("features","target")
data.show(5)

+--------------------+------+
|            features|target|
+--------------------+------+
|[63.0,1.0,3.0,145...|     1|
|[37.0,1.0,2.0,130...|     1|
|[41.0,0.0,1.0,130...|     1|
|[56.0,1.0,1.0,120...|     1|
|[57.0,0.0,0.0,120...|     1|
+--------------------+------+
only showing top 5 rows



In [9]:
train_data, test_data = data.randomSplit([0.8,0.2], seed=42)

In [10]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(labelCol="target")
model = lr.fit(train_data)

In [11]:
predictions = model.transform(test_data)
predictions.select("target","prediction","probability").show(10)

+------+----------+--------------------+
|target|prediction|         probability|
+------+----------+--------------------+
|     1|       1.0|[0.20622881833055...|
|     1|       1.0|[0.04023440726126...|
|     1|       1.0|[0.35583435765664...|
|     1|       1.0|[0.15017416947970...|
|     1|       1.0|[0.07087714487496...|
|     1|       1.0|[0.06725630522706...|
|     1|       1.0|[0.06366629547004...|
|     1|       1.0|[0.07900598898263...|
|     1|       1.0|[0.02516440287505...|
|     1|       1.0|[0.04700969851101...|
+------+----------+--------------------+
only showing top 10 rows



In [12]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)
print("Accuracy =", accuracy)

Accuracy = 0.8269230769230769


In [ ]:
#QUESTION 2
#Build a Clustering Model with Spark

In [14]:
url = "/content/iris.csv"
pdf = pd.read_csv(url)
df = spark.createDataFrame(pdf)
df.show()

+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|species|
+------------+-----------+------------+-----------+-------+
|         5.1|        3.5|         1.4|        0.2| setosa|
|         4.9|        3.0|         1.4|        0.2| setosa|
|         4.7|        3.2|         1.3|        0.2| setosa|
|         4.6|        3.1|         1.5|        0.2| setosa|
|         5.0|        3.6|         1.4|        0.2| setosa|
|         5.4|        3.9|         1.7|        0.4| setosa|
|         4.6|        3.4|         1.4|        0.3| setosa|
|         5.0|        3.4|         1.5|        0.2| setosa|
|         4.4|        2.9|         1.4|        0.2| setosa|
|         4.9|        3.1|         1.5|        0.1| setosa|
|         5.4|        3.7|         1.5|        0.2| setosa|
|         4.8|        3.4|         1.6|        0.2| setosa|
|         4.8|        3.0|         1.4|        0.1| setosa|
|         4.3|        3.0|         1.1| 

In [15]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["sepal_length","sepal_width","petal_length","petal_width"],
    outputCol="features"
)

data = assembler.transform(df)

In [16]:
from pyspark.ml.clustering import KMeans

kmeans = KMeans(k=3, seed=1)
model = kmeans.fit(data)

In [17]:
predictions = model.transform(data)
predictions.select("features","prediction").show(10)

print("Cluster Centers:")
for c in model.clusterCenters():
    print(c)

+-----------------+----------+
|         features|prediction|
+-----------------+----------+
|[5.1,3.5,1.4,0.2]|         1|
|[4.9,3.0,1.4,0.2]|         1|
|[4.7,3.2,1.3,0.2]|         1|
|[4.6,3.1,1.5,0.2]|         1|
|[5.0,3.6,1.4,0.2]|         1|
|[5.4,3.9,1.7,0.4]|         1|
|[4.6,3.4,1.4,0.3]|         1|
|[5.0,3.4,1.5,0.2]|         1|
|[4.4,2.9,1.4,0.2]|         1|
|[4.9,3.1,1.5,0.1]|         1|
+-----------------+----------+
only showing top 10 rows

Cluster Centers:
[5.88360656 2.74098361 4.38852459 1.43442623]
[5.006 3.428 1.462 0.246]
[6.85384615 3.07692308 5.71538462 2.05384615]


In [ ]:
#QUESTION 3 — RECOMMENDATION ENGINE (MovieLens ALS)

In [24]:
from pyspark.ml.recommendation import ALS
from pyspark.sql import Row

ratings = spark.createDataFrame([
    Row(userId=1, itemId=101, rating=5),
    Row(userId=1, itemId=102, rating=3),
    Row(userId=2, itemId=101, rating=4),
    Row(userId=2, itemId=103, rating=2),
    Row(userId=3, itemId=102, rating=4),
    Row(userId=3, itemId=103, rating=5)
])

als = ALS(
    maxIter=5,
    regParam=0.01,
    userCol="userId",
    itemCol="itemId",
    ratingCol="rating",
    coldStartStrategy="drop"
)

model_als = als.fit(ratings)


user_recs = model_als.recommendForAllUsers(2)
item_recs = model_als.recommendForAllItems(2)

print("User Recommendations:")
user_recs.show(truncate=False)

print("Item Recommendations:")
item_recs.show(truncate=False)

User Recommendations:
+------+-----------------------------------+
|userId|recommendations                    |
+------+-----------------------------------+
|1     |[{101, 4.997869}, {103, 4.933512}] |
|2     |[{101, 3.996679}, {103, 2.0013847}]|
|3     |[{103, 5.000776}, {102, 3.9933474}]|
+------+-----------------------------------+

Item Recommendations:
+------+-------------------------------+
|itemId|recommendations                |
+------+-------------------------------+
|101   |[{1, 4.997869}, {2, 3.996679}] |
|102   |[{3, 3.9933474}, {1, 2.997328}]|
|103   |[{3, 5.000776}, {1, 4.933512}] |
+------+-------------------------------+

